In [1]:
try:
    from google.colab import drive  # Colab only
except Exception:
    drive = None
import os as _os
if drive is not None and not _os.path.ismount("/content/drive"):
    drive.mount("/content/drive")  # [sanitized] optional; data paths are relative to DrugReview_ROOT

import sys
from pathlib import Path
import os
import pandas as pd

DrugReview_ROOT = Path(".")
sys.path.append(str(DrugReview_ROOT))



RAW_DATA_FILE = "PureText/drugsCom_generalized_with_ai_labels_mini.csv"
DATA_PATH = os.path.join(DrugReview_ROOT, "data", RAW_DATA_FILE)  # or os.path.join(PROJECT_PATH, RAW_DATA_FILE)
df = pd.read_csv(DATA_PATH)

Mounted at /content/drive


In [ ]:
import numpy as np
import pandas as pd

from sklearn.metrics import (
    confusion_matrix, classification_report, cohen_kappa_score, accuracy_score
)

# ---------------------------
# Config
# ---------------------------
HUMAN_COL = "sentiment_5"         # human 5-class
AI_HARD_COL = "ai_sentiment_5"    # GPT hard 5-class
AI_RATING_COL = "ai_rating_10"    # GPT 1..10

PROB_COLS = [
    "ai_prob_very_negative",
    "ai_prob_negative",
    "ai_prob_neutral",
    "ai_prob_positive",
    "ai_prob_very_positive",
]

ORDER5 = ["very_negative", "negative", "neutral", "positive", "very_positive"]
LABEL2IDX = {lab:i for i,lab in enumerate(ORDER5)}
IDX2LABEL = {i:lab for lab,i in LABEL2IDX.items()}

# ---------------------------
# Helpers
# ---------------------------
def norm5(x):
    if pd.isna(x):
        return np.nan
    return (str(x).strip()
            .lower()
            .replace(" ", "_")
            .replace("-", "_"))

def rating10_to_sent5(r):
    if pd.isna(r):
        return np.nan
    r = int(r)
    if r <= 2:  return "very_negative"
    if r <= 4:  return "negative"
    if r <= 6:  return "neutral"
    if r <= 8:  return "positive"
    return "very_positive"

def defensive_renorm(p, eps=1e-12):
    """Clip + renormalize rows to sum to 1 (numerical tolerance)."""
    p = np.clip(p, eps, 1.0)
    row_sums = p.sum(axis=1, keepdims=True)
    return p / row_sums

def entropy_norm(p):
    """Normalized Shannon entropy in [0,1] (natural log)."""
    K = p.shape[1]
    H = -np.sum(p * np.log(p), axis=1)
    return H / np.log(K)

def multiclass_brier(p, y_idx):
    """Mean sum_k (p_k - 1[y=k])^2."""
    n, K = p.shape
    y_onehot = np.zeros((n, K), dtype=float)
    y_onehot[np.arange(n), y_idx] = 1.0
    return np.mean(np.sum((p - y_onehot)**2, axis=1))

# ---------------------------
# 1) Normalize labels
# ---------------------------
df[HUMAN_COL] = df[HUMAN_COL].map(norm5)
df[AI_HARD_COL] = df[AI_HARD_COL].map(norm5)

# rating-derived 5-class from GPT rating_10
df["ai_sent5_from_rating10"] = df[AI_RATING_COL].apply(rating10_to_sent5)

# ---------------------------
# 2) Build probability matrix + argmax label
# ---------------------------
missing = [c for c in PROB_COLS if c not in df.columns]
assert len(missing) == 0, f"Missing probability columns: {missing}"

p_raw = df[PROB_COLS].astype(float).to_numpy()
p = defensive_renorm(p_raw)

df["ai_prob_argmax"] = np.argmax(p, axis=1)
df["ai_sent5_from_probs"] = df["ai_prob_argmax"].map(IDX2LABEL)

df["ai_pmax"] = p.max(axis=1)
df["ai_entropy_norm"] = entropy_norm(p)

# ---------------------------
# 3) HARD-LABEL AGREEMENT (human vs different AI hard labels)
# ---------------------------
def hard_agreement(y_true, y_pred, name):
    mask = y_true.notna() & y_pred.notna()
    yt = y_true[mask].astype(str)
    yp = y_pred[mask].astype(str)

    acc = accuracy_score(yt, yp)
    qwk = cohen_kappa_score(yt, yp, labels=ORDER5, weights="quadratic")

    cm = confusion_matrix(yt, yp, labels=ORDER5)
    cm_df = pd.DataFrame(cm, index=[f"T:{c}" for c in ORDER5], columns=[f"P:{c}" for c in ORDER5])

    print(f"\n=== {name} ===")
    print(f"N = {len(yt)}")
    print(f"Exact agreement (accuracy): {acc:.4f}")
    print(f"Quadratic weighted kappa:   {qwk:.4f}")
    print("\nConfusion matrix (counts):")
    display(cm_df)
    print("\nClassification report:")
    print(classification_report(yt, yp, labels=ORDER5, zero_division=0))

# A) Human vs GPT hard label
hard_agreement(df[HUMAN_COL], df[AI_HARD_COL], "Human sentiment_5 vs AI hard ai_sentiment_5")

# B) Human vs GPT rating10-binned label
hard_agreement(df[HUMAN_COL], df["ai_sent5_from_rating10"], "Human sentiment_5 vs AI (ai_rating_10 -> 5-class)")

# C) Human vs Prob-argmax label
hard_agreement(df[HUMAN_COL], df["ai_sent5_from_probs"], "Human sentiment_5 vs AI (prob argmax)")

# D) Internal consistency: GPT hard vs prob-argmax
hard_agreement(df[AI_HARD_COL], df["ai_sent5_from_probs"], "AI hard ai_sentiment_5 vs AI prob-argmax")

# ---------------------------
# 4) SOFT-LABEL METRICS (human vs probability vector)
# ---------------------------
mask_soft = df[HUMAN_COL].notna()
y_true = df.loc[mask_soft, HUMAN_COL].map(LABEL2IDX).astype(int).to_numpy()
p_soft = p[mask_soft.to_numpy()]

# Prob assigned to the human class
p_true = p_soft[np.arange(len(y_true)), y_true]
mean_p_true = float(np.mean(p_true))

# NLL of the human class (cross-entropy on observed label)
nll = float(np.mean(-np.log(np.clip(p_true, 1e-12, 1.0))))

# Multiclass Brier
brier = float(multiclass_brier(p_soft, y_true))

# Ordinal expected class index + error
idx = np.arange(5)
exp_idx = p_soft @ idx
mae = float(np.mean(np.abs(exp_idx - y_true)))
rmse = float(np.sqrt(np.mean((exp_idx - y_true)**2)))

# Expected absolute ordinal error (W1 to one-hot along ordinal axis)
# (equivalent here to E[|K - y|] under p)
w1 = float(np.mean(np.sum(p_soft * np.abs(idx[None, :] - y_true[:, None]), axis=1)))

print("\n=== Soft-label diagnostics (Human vs AI prob vector) ===")
print(f"N = {len(y_true)}")
print(f"Mean P(true human class): {mean_p_true:.4f}")
print(f"NLL (cross-entropy):     {nll:.4f}")
print(f"Brier (multiclass):      {brier:.4f}")
print(f"Ordinal MAE (E[idx]-y):  {mae:.4f}")
print(f"Ordinal RMSE:            {rmse:.4f}")
print(f"Expected |ordinal error|:{w1:.4f}")

# ---------------------------
# 5) Stratify agreement by confidence / uncertainty (optional but very informative)
# ---------------------------
tmp = df.loc[mask_soft, [HUMAN_COL, "ai_sent5_from_probs", "ai_pmax", "ai_entropy_norm"]].copy()
tmp["correct_prob_argmax"] = (tmp[HUMAN_COL] == tmp["ai_sent5_from_probs"])

# bins for pmax and entropy
tmp["pmax_bin"] = pd.cut(tmp["ai_pmax"], bins=[0, .4, .6, .8, .9, 1.0], include_lowest=True)
tmp["ent_bin"]  = pd.cut(tmp["ai_entropy_norm"], bins=[0, .2, .4, .6, .8, 1.0], include_lowest=True)

summary_pmax = tmp.groupby("pmax_bin")["correct_prob_argmax"].agg(["count", "mean"]).rename(columns={"mean":"acc"})
summary_ent  = tmp.groupby("ent_bin")["correct_prob_argmax"].agg(["count", "mean"]).rename(columns={"mean":"acc"})

print("\nAgreement vs p_max bins (Human vs prob-argmax):")
display(summary_pmax)

print("\nAgreement vs entropy bins (Human vs prob-argmax):")
display(summary_ent)


=== Human sentiment_5 vs AI hard ai_sentiment_5 ===
N = 28755
Exact agreement (accuracy): 0.5019
Quadratic weighted kappa:   0.7997

Confusion matrix (counts):


,P:very_negative,P:negative,P:neutral,P:positive,P:very_positive
T:very_negative,4218,1063,112,22,8
T:negative,859,1128,313,30,2
T:neutral,414,1173,927,215,18
T:positive,123,606,1692,1940,420
T:very_positive,122,294,1397,5440,6219



Classification report:
               precision    recall  f1-score   support

very_negative       0.74      0.78      0.76      5423
     negative       0.26      0.48      0.34      2332
      neutral       0.21      0.34      0.26      2747
     positive       0.25      0.41      0.31      4781
very_positive       0.93      0.46      0.62     13472

     accuracy                           0.50     28755
    macro avg       0.48      0.49      0.46     28755
 weighted avg       0.66      0.50      0.54     28755


=== Human sentiment_5 vs AI (ai_rating_10 -> 5-class) ===
N = 28755
Exact agreement (accuracy): 0.5019
Quadratic weighted kappa:   0.7997

Confusion matrix (counts):


,P:very_negative,P:negative,P:neutral,P:positive,P:very_positive
T:very_negative,4218,1063,112,22,8
T:negative,859,1128,313,30,2
T:neutral,414,1173,927,215,18
T:positive,123,606,1692,1940,420
T:very_positive,122,294,1397,5440,6219



Classification report:
               precision    recall  f1-score   support

very_negative       0.74      0.78      0.76      5423
     negative       0.26      0.48      0.34      2332
      neutral       0.21      0.34      0.26      2747
     positive       0.25      0.41      0.31      4781
very_positive       0.93      0.46      0.62     13472

     accuracy                           0.50     28755
    macro avg       0.48      0.49      0.46     28755
 weighted avg       0.66      0.50      0.54     28755


=== Human sentiment_5 vs AI (prob argmax) ===
N = 28755
Exact agreement (accuracy): 0.5348
Quadratic weighted kappa:   0.8001

Confusion matrix (counts):


...


Classification report:
               precision    recall  f1-score   support

very_negative       0.74      0.99      0.85      5736
     negative       0.64      0.54      0.58      4264
      neutral       0.92      0.16      0.27      4441
     positive       0.61      0.71      0.66      7647
very_positive       0.72      0.85      0.78      6667

     accuracy                           0.69     28755
    macro avg       0.73      0.65      0.63     28755
 weighted avg       0.71      0.69      0.65     28755


=== Soft-label diagnostics (Human vs AI prob vector) ===
N = 28755
Mean P(true human class): 0.4049
NLL (cross-entropy):     1.5332
Brier (multiclass):      0.5810
Ordinal MAE (E[idx]-y):  0.7464
Ordinal RMSE:            0.9171
Expected |ordinal error|:0.8831

Agreement vs p_max bins (Human vs prob-argmax):


/tmp/ipython-input-1095427977.py:172: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  summary_pmax = tmp.groupby("pmax_bin")["correct_prob_argmax"].agg(["count", "mean"]).rename(columns={"mean":"acc"})
/tmp/ipython-input-1095427977.py:173: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  summary_ent  = tmp.groupby("ent_bin")["correct_prob_argmax"].agg(["count", "mean"]).rename(columns={"mean":"acc"})


,count,acc
pmax_bin,,
"(-0.001, 0.4]",11091,0.300063
"(0.4, 0.6]",14361,0.645220
"(0.6, 0.8]",2547,0.833922
"(0.8, 0.9]",660,0.872727
"(0.9, 1.0]",96,0.864583



Agreement vs entropy bins (Human vs prob-argmax):


,count,acc
ent_bin,,
"(-0.001, 0.2]",96,0.864583
"(0.2, 0.4]",2228,0.860413
"(0.4, 0.6]",11077,0.738106
"(0.6, 0.8]",9450,0.378307
"(0.8, 1.0]",5898,0.275687


In [ ]:
import pandas as pd
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, cohen_kappa_score

# --- helpers ---
LABELS_5 = ["very_negative", "negative", "neutral", "positive", "very_positive"]

def normalize_5class(x):
    """Make ai_sentiment_5 robust to casing / spaces / hyphens."""
    if pd.isna(x):
        return pd.NA
    s = str(x).strip().lower().replace("-", "_").replace(" ", "_")
    # common variants (optional)
    mapping = {
        "verynegative": "very_negative",
        "verypositive": "very_positive",
        "vneg": "very_negative",
        "vpos": "very_positive",
    }
    return mapping.get(s, s)

def rating10_to_5class(r):
    """Map 1–10 rating to 5 ordinal bins: 1-2,3-4,5-6,7-8,9-10."""
    if pd.isna(r):
        return pd.NA
    try:
        rr = int(float(r))
    except Exception:
        return pd.NA
    if rr < 1 or rr > 10:
        return pd.NA
    if rr <= 2:
        return "very_negative"
    elif rr <= 4:
        return "negative"
    elif rr <= 6:
        return "neutral"
    elif rr <= 8:
        return "positive"
    else:
        return "very_positive"

# --- main ---
# df = ...  # your dataframe containing ai_sentiment_5 and ai_rating_10

df = df.copy()
df["ai_sentiment_5_norm"] = df["ai_sentiment_5"].apply(normalize_5class)
df["ai_sent5_from_rating10"] = df["ai_rating_10"].apply(rating10_to_5class)

sub = df[["ai_sentiment_5_norm", "ai_sent5_from_rating10"]].dropna()

y1 = sub["ai_sentiment_5_norm"]
y2 = sub["ai_sent5_from_rating10"]

acc = accuracy_score(y1, y2)
kappa_q = cohen_kappa_score(y1, y2, labels=LABELS_5, weights="quadratic")
cm = confusion_matrix(y1, y2, labels=LABELS_5)

print(f"N = {len(sub)}")
print(f"Exact agreement (accuracy): {acc:.4f}")
print(f"Quadratic weighted kappa:   {kappa_q:.4f}\n")

print("Confusion matrix (counts):")
print(pd.DataFrame(cm, index=[f"T:{c}" for c in LABELS_5], columns=[f"P:{c}" for c in LABELS_5]))
print("\nClassification report:")
print(classification_report(y1, y2, labels=LABELS_5, digits=2))

N = 28755
Exact agreement (accuracy): 1.0000
Quadratic weighted kappa:   1.0000

Confusion matrix (counts):
                 P:very_negative  P:negative  P:neutral  P:positive  \
T:very_negative             5736           0          0           0   
T:negative                     0        4264          0           0   
T:neutral                      0           0       4441           0   
T:positive                     0           0          0        7647   
T:very_positive                0           0          0           0   

                 P:very_positive  
T:very_negative                0  
T:negative                     0  
T:neutral                      0  
T:positive                     0  
T:very_positive             6667  

Classification report:
               precision    recall  f1-score   support

very_negative       1.00      1.00      1.00      5736
     negative       1.00      1.00      1.00      4264
      neutral       1.00      1.00      1.00      4441
     posit

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, cohen_kappa_score

# ---------------------------
# REQUIREMENTS from your existing pipeline:
# df contains:
#   HUMAN_COL (sentiment_5)
#   AI_HARD_COL (ai_sentiment_5)
#   ai_sent5_from_probs (prob argmax label, 5-class)
#   ai_pmax (max prob)
# ---------------------------

ORDER5 = ["very_negative", "negative", "neutral", "positive", "very_positive"]
VAL5 = {"very_negative": -2, "negative": -1, "neutral": 0, "positive": 1, "very_positive": 2}

def norm5(x):
    if pd.isna(x):
        return np.nan
    return (str(x).strip().lower().replace(" ", "_").replace("-", "_"))

def label_distribution(s, order=ORDER5):
    s = s.dropna().astype(str)
    counts = s.value_counts().reindex(order).fillna(0).astype(int)
    props = counts / counts.sum() if counts.sum() > 0 else counts.astype(float)
    return pd.DataFrame({"count": counts, "prop": props})

def jensen_shannon(p, q, eps=1e-12):
    p = np.asarray(p, dtype=float) + eps
    q = np.asarray(q, dtype=float) + eps
    p = p / p.sum()
    q = q / q.sum()
    m = 0.5 * (p + q)
    kl_pm = np.sum(p * np.log(p / m))
    kl_qm = np.sum(q * np.log(q / m))
    return float(np.sqrt(0.5 * (kl_pm + kl_qm)))

def ordinal_error_summary(y_true, y_pred):
    yt = y_true.map(norm5)
    yp = y_pred.map(norm5)
    m = yt.notna() & yp.notna()
    yt = yt[m].astype(str)
    yp = yp[m].astype(str)

    yv = yt.map(VAL5).to_numpy()
    pv = yp.map(VAL5).to_numpy()

    abs_err = np.abs(pv - yv)
    out = {
        "N": int(m.sum()),
        "mean_abs_ord_err": float(abs_err.mean()),
        "ord_rmse": float(np.sqrt(np.mean((pv - yv)**2))),
        "within_1_class": float((abs_err <= 1).mean()),
    }
    return out

def bootstrap_ci_acc_qwk(y_true, y_pred, B=2000, seed=0):
    rng = np.random.default_rng(seed)
    yt = y_true.map(norm5)
    yp = y_pred.map(norm5)
    m = yt.notna() & yp.notna()
    yt = yt[m].astype(str).to_numpy()
    yp = yp[m].astype(str).to_numpy()
    n = len(yt)

    accs = np.empty(B, dtype=float)
    qwks = np.empty(B, dtype=float)
    for b in range(B):
        idx = rng.integers(0, n, size=n)
        ytb, ypb = yt[idx], yp[idx]
        accs[b] = accuracy_score(ytb, ypb)
        qwks[b] = cohen_kappa_score(ytb, ypb, labels=ORDER5, weights="quadratic")

    def ci(x):
        return (float(np.quantile(x, 0.025)), float(np.quantile(x, 0.975)))

    return {
        "N": n,
        "acc_mean": float(accs.mean()),
        "acc_ci95": ci(accs),
        "qwk_mean": float(qwks.mean()),
        "qwk_ci95": ci(qwks),
    }

def ece_from_pmax(y_true, y_pred, pmax, n_bins=10):
    y_true = np.asarray([norm5(x) for x in y_true])
    y_pred = np.asarray([norm5(x) for x in y_pred])
    pmax = np.asarray(pmax, dtype=float)

    m = pd.notna(y_true) & pd.notna(y_pred) & pd.notna(pmax)
    y_true = y_true[m]
    y_pred = y_pred[m]
    pmax = pmax[m]

    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    rows = []

    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        sel = (pmax > lo) & (pmax <= hi) if i > 0 else (pmax >= lo) & (pmax <= hi)
        if sel.sum() == 0:
            rows.append([f"({lo:.1f},{hi:.1f}]", 0, np.nan, np.nan, np.nan])
            continue
        acc_b = float((y_true[sel] == y_pred[sel]).mean())
        conf_b = float(pmax[sel].mean())
        w = float(sel.mean())
        ece += w * abs(acc_b - conf_b)
        rows.append([f"({lo:.1f},{hi:.1f}]", int(sel.sum()), acc_b, conf_b, abs(acc_b - conf_b)])

    tab = pd.DataFrame(rows, columns=["bin", "count", "acc", "mean_conf", "|acc-conf|"])
    return float(ece), tab

# ---------------------------
# IMPLEMENTATION (runs now)
# ---------------------------
HUMAN_COL = "sentiment_5"
AI_HARD_COL = "ai_sentiment_5"
AI_ARGMAX_COL = "ai_sent5_from_probs"
PMAX_COL = "ai_pmax"

# Normalize once
df[HUMAN_COL] = df[HUMAN_COL].map(norm5)
df[AI_HARD_COL] = df[AI_HARD_COL].map(norm5)
df[AI_ARGMAX_COL] = df[AI_ARGMAX_COL].map(norm5)

print("\n==============================")
print("EXTRA LABEL EVAL DIAGNOSTICS")
print("==============================")

# (A) Marginal distributions + JS distance
for pred_col, name in [(AI_HARD_COL, "AI hard"), (AI_ARGMAX_COL, "AI prob-argmax")]:
    tab_h = label_distribution(df[HUMAN_COL])
    tab_a = label_distribution(df[pred_col])
    js = jensen_shannon(tab_h["prop"].values, tab_a["prop"].values)

    print(f"\n--- Marginal shift: Human vs {name} ---")
    display(pd.concat({"Human": tab_h, name: tab_a}, axis=1))
    print("JS distance:", round(js, 4))

# (B) Ordinal-distance error for hard labels (interpretable)
for pred_col, name in [(AI_HARD_COL, "Human vs AI hard"),
                       (AI_ARGMAX_COL, "Human vs AI prob-argmax")]:
    out = ordinal_error_summary(df[HUMAN_COL], df[pred_col])
    print(f"\n--- Ordinal error: {name} ---")
    for k, v in out.items():
        print(f"{k}: {v:.4f}" if isinstance(v, float) else f"{k}: {v}")

# (C) Bootstrap CI for accuracy + QWK (hard agreement)
for pred_col, name in [(AI_HARD_COL, "Human vs AI hard"),
                       (AI_ARGMAX_COL, "Human vs AI prob-argmax")]:
    ci = bootstrap_ci_acc_qwk(df[HUMAN_COL], df[pred_col], B=2000, seed=123)
    print(f"\n--- Bootstrap 95% CI: {name} ---")
    print("N:", ci["N"])
    print(f"Accuracy: {ci['acc_mean']:.4f}  (95% CI {ci['acc_ci95'][0]:.4f}, {ci['acc_ci95'][1]:.4f})")
    print(f"QWK:      {ci['qwk_mean']:.4f}  (95% CI {ci['qwk_ci95'][0]:.4f}, {ci['qwk_ci95'][1]:.4f})")

# (D) Calibration via ECE (for prob-argmax vs human)
if PMAX_COL in df.columns:
    mask = df[HUMAN_COL].notna() & df[AI_ARGMAX_COL].notna() & df[PMAX_COL].notna()
    ece, tab = ece_from_pmax(df.loc[mask, HUMAN_COL],
                             df.loc[mask, AI_ARGMAX_COL],
                             df.loc[mask, PMAX_COL],
                             n_bins=10)
    print("\n--- Calibration: ECE (prob-argmax using pmax) ---")
    print("ECE:", round(ece, 4))
    display(tab)
else:
    print("\n[Skip] ai_pmax not found; cannot compute ECE.")


EXTRA LABEL EVAL DIAGNOSTICS

--- Marginal shift: Human vs AI hard ---


Human           AI hard          
               count      prop   count      prop
very_negative   5423  0.188593    5736  0.199478
negative        2332  0.081099    4264  0.148287
neutral         2747  0.095531    4441  0.154443
positive        4781  0.166267    7647  0.265936
very_positive  13472  0.468510    6667  0.231855

JS distance: 0.1864

--- Marginal shift: Human vs AI prob-argmax ---


Human           AI prob-argmax          
               count      prop          count      prop
very_negative   5423  0.188593           7667  0.266632
negative        2332  0.081099           3586  0.124709
neutral         2747  0.095531            764  0.026569
positive        4781  0.166267           8823  0.306834
very_positive  13472  0.468510           7915  0.275256

JS distance: 0.1983

--- Ordinal error: Human vs AI hard ---
N: 28755
mean_abs_ord_err: 0.6320
ord_rmse: 0.9785
within_1_class: 0.8905

--- Ordinal error: Human vs AI prob-argmax ---
N: 28755
mean_abs_ord_err: 0.6221
ord_rmse: 1.0212
within_1_class: 0.8884

--- Bootstrap 95% CI: Human vs AI hard ---
N: 28755
Accuracy: 0.5019  (95% CI 0.4961, 0.5075)
QWK:      0.7997  (95% CI 0.7948, 0.8043)

--- Bootstrap 95% CI: Human vs AI prob-argmax ---
N: 28755
Accuracy: 0.5348  (95% CI 0.5290, 0.5407)
QWK:      0.8001  (95% CI 0.7950, 0.8053)

--- Calibration: ECE (prob-argmax using pmax) ---
ECE: 0.1001


,bin,count,acc,mean_conf,|acc-conf|
0,"(0.0,0.1]",0,NaN,NaN,NaN
1,"(0.1,0.2]",6,0.000000,0.200000,0.200000
2,"(0.2,0.3]",2399,0.225094,0.299797,0.074703
3,"(0.3,0.4]",8686,0.320976,0.389284,0.068308
4,"(0.4,0.5]",10796,0.607632,0.491130,0.116502
5,"(0.5,0.6]",3566,0.758833,0.598552,0.160282
6,"(0.6,0.7]",1518,0.814888,0.697273,0.117615
7,"(0.7,0.8]",1028,0.862840,0.798235,0.064606
8,"(0.8,0.9]",660,0.872727,0.899450,0.026723
9,"(0.9,1.0]",96,0.864583,1.000000,0.135417


In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

# ---------------------------
# Helpers
# ---------------------------
ORDER5 = ["very_negative", "negative", "neutral", "positive", "very_positive"]
PROB_COLS_DEFAULT = [
    "ai_prob_very_negative",
    "ai_prob_negative",
    "ai_prob_neutral",
    "ai_prob_positive",
    "ai_prob_very_positive",
]

def norm5(x):
    if pd.isna(x):
        return np.nan
    return (str(x).strip()
            .lower()
            .replace(" ", "_")
            .replace("-", "_"))

def defensive_renorm(P, eps=1e-12):
    P = np.clip(P, eps, 1.0)
    rs = P.sum(axis=1, keepdims=True)
    return P / rs

def entropy_norm(P):
    # normalized Shannon entropy in [0,1]
    K = P.shape[1]
    H = -np.sum(P * np.log(P), axis=1)
    return H / np.log(K)

def fit_disagreement_logit(
    df,
    human_col="sentiment_5",
    ai_col="ai_sentiment_5",
    review_col="review",
    rating_col="rating",
    prob_cols=PROB_COLS_DEFAULT,
    use_entropy=True,                 # if False, uses pmax instead
    add_eff_saf=False,                # optionally add efficacy/safety if present
    robust="HC3",                     # "HC3" (default) or None
    cluster_col=None,                 # e.g., "drugName" if you want cluster-robust SEs
):
    d = df.copy()

    # --- normalize labels ---
    d[human_col] = d[human_col].map(norm5)
    d[ai_col] = d[ai_col].map(norm5)

    # --- keep rows with both labels ---
    mask = d[human_col].notna() & d[ai_col].notna()
    d = d.loc[mask].copy()

    # --- outcome: disagreement indicator ---
    d["disagree"] = (d[human_col] != d[ai_col]).astype(int)

    # --- review length ---
    if review_col in d.columns:
        d["review_len"] = d[review_col].astype(str).str.len()
    else:
        d["review_len"] = np.nan

    # --- rating numeric ---
    if rating_col in d.columns:
        d["rating_num"] = pd.to_numeric(d[rating_col], errors="coerce")
    else:
        d["rating_num"] = np.nan

    # --- uncertainty features from probabilities ---
    missing = [c for c in prob_cols if c not in d.columns]
    if missing:
        raise ValueError(f"Missing prob columns needed for uncertainty: {missing}")

    P = d[prob_cols].astype(float).to_numpy()
    P = defensive_renorm(P)
    d["pmax"] = P.max(axis=1)
    d["entropy_norm"] = entropy_norm(P)

    # --- choose uncertainty regressor ---
    unc = "entropy_norm" if use_entropy else "pmax"

    # --- build formula ---
    base_terms = [unc, "review_len", "rating_num"]

    # optionally add efficacy/safety as categorical controls if columns exist
    if add_eff_saf:
        for col in ["ai_efficacy", "ai_safety"]:
            if col in d.columns:
                base_terms.append(f"C({col})")

    # drop rows with missing covariates used in model
    needed_cols = ["disagree", unc, "review_len", "rating_num"]
    d_model = d.dropna(subset=needed_cols).copy()

    formula = "disagree ~ " + " + ".join(base_terms)

    # --- fit logit ---
    model = smf.logit(formula, data=d_model)
    if cluster_col is not None and cluster_col in d_model.columns:
        res = model.fit(disp=False, cov_type="cluster", cov_kwds={"groups": d_model[cluster_col]})
    elif robust is not None:
        res = model.fit(disp=False, cov_type=robust)
    else:
        res = model.fit(disp=False)

    return res, formula, d_model


def summarize_odds_ratios(res):
    """Pretty OR table with 95% CI."""
    params = res.params
    se = res.bse
    z = 1.96
    out = pd.DataFrame({
        "coef": params,
        "se": se,
        "OR": np.exp(params),
        "OR_2.5%": np.exp(params - z*se),
        "OR_97.5%": np.exp(params + z*se),
        "p": res.pvalues
    })
    return out.sort_values("p")


# ---------------------------
# IMPLEMENTATION ON df
# ---------------------------

# 1) Disagreement ~ entropy + controls
res_ent, formula_ent, d_used_ent = fit_disagreement_logit(
    df,
    human_col="sentiment_5",
    ai_col="ai_sentiment_5",
    review_col="review",
    rating_col="rating",
    use_entropy=True,
    add_eff_saf=False,   # flip to True if you want to control for ai_efficacy/ai_safety
    robust="HC3",
    cluster_col=None,    # e.g. "drugName" if you want clustered SE
)
print("MODEL (entropy):", formula_ent)
print(res_ent.summary())
display(summarize_odds_ratios(res_ent).head(15))

# 2) Disagreement ~ pmax + controls
res_pmax, formula_pmax, d_used_pmax = fit_disagreement_logit(
    df,
    human_col="sentiment_5",
    ai_col="ai_sentiment_5",
    review_col="review",
    rating_col="rating",
    use_entropy=False,   # uses pmax
    add_eff_saf=False,
    robust="HC3",
    cluster_col=None,
)
print("\nMODEL (pmax):", formula_pmax)
print(res_pmax.summary())
display(summarize_odds_ratios(res_pmax).head(15))

MODEL (entropy): disagree ~ entropy_norm + review_len + rating_num
                           Logit Regression Results                           
Dep. Variable:               disagree   No. Observations:                28755
Model:                          Logit   Df Residuals:                    28751
Method:                           MLE   Df Model:                            3
Date:                Sat, 20 Dec 2025   Pseudo R-squ.:                  0.1972
Time:                        23:47:34   Log-Likelihood:                -16002.
converged:                       True   LL-Null:                       -19931.
Covariance Type:                  HC3   LLR p-value:                     0.000
                   coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept       -5.7285      0.080    -71.815      0.000      -5.885      -5.572
entropy_norm     6.6914      0.098     68.316      0.000  

,coef,se,OR,OR_2.5%,OR_97.5%,p
Intercept,-5.728492,0.079768,0.003252,0.002781,0.003802,0.000000
entropy_norm,6.691386,0.097948,805.437900,664.748459,975.903295,0.000000
rating_num,0.185884,0.003989,1.204282,1.194904,1.213734,0.000000
review_len,-0.000220,0.000058,0.999780,0.999666,0.999894,0.000154



MODEL (pmax): disagree ~ pmax + review_len + rating_num
                           Logit Regression Results                           
Dep. Variable:               disagree   No. Observations:                28755
Model:                          Logit   Df Residuals:                    28751
Method:                           MLE   Df Model:                            3
Date:                Sat, 20 Dec 2025   Pseudo R-squ.:                  0.1868
Time:                        23:47:35   Log-Likelihood:                -16209.
converged:                       True   LL-Null:                       -19931.
Covariance Type:                  HC3   LLR p-value:                     0.000
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      3.2200      0.078     41.052      0.000       3.066       3.374
pmax          -9.0191      0.146    -61.601      0.000      -9.306      -8

,coef,se,OR,OR_2.5%,OR_97.5%,p
Intercept,3.219998,0.078437,25.028074,21.461501,29.187357,0.000000
pmax,-9.019062,0.146410,0.000121,0.000091,0.000161,0.000000
rating_num,0.166311,0.003927,1.180940,1.171885,1.190064,0.000000
review_len,-0.000068,0.000057,0.999932,0.999820,1.000045,0.239363


In [ ]:
import numpy as np
import pandas as pd

# ---- config ----
HUMAN_RATING = "rating"          # original user rating column (string/int)
HUMAN5 = "sentiment_5"           # human 5-class
AI5 = "ai_sentiment_5"           # AI hard 5-class
AI_R10 = "ai_rating_10"          # AI 1..10

def norm5(x):
    if pd.isna(x): return np.nan
    return str(x).strip().lower().replace(" ", "_").replace("-", "_")

def rating10_to_sent5(r):
    if pd.isna(r): return np.nan
    r = int(r)
    if r <= 2:  return "very_negative"
    if r <= 4:  return "negative"
    if r <= 6:  return "neutral"
    if r <= 8:  return "positive"
    return "very_positive"

# =========================
# 1) Check human rating_num
# =========================
df["rating_num"] = pd.to_numeric(df[HUMAN_RATING], errors="coerce")

print("Human rating_num summary:")
print(df["rating_num"].describe())
print("\nHuman rating_num invalid (NaN or outside 1..10):",
      int(df["rating_num"].isna().sum() + ((df["rating_num"]<1)|(df["rating_num"]>10)).sum()))
print("\nTop raw values of `rating` (to confirm it's the original column):")
print(df[HUMAN_RATING].value_counts(dropna=False).head(15))

# If you have a `date` column and want to check parsing issues:
if "date" in df.columns:
    print("\nDate col dtype:", df["date"].dtype)

# ======================
# 2) Check AI ai_rating_10
# ======================
df["ai_rating10_num"] = pd.to_numeric(df[AI_R10], errors="coerce")

print("\nAI ai_rating_10 summary:")
print(df["ai_rating10_num"].describe())
print("\nAI ai_rating_10 invalid (NaN or outside 1..10):",
      int(df["ai_rating10_num"].isna().sum() + ((df["ai_rating10_num"]<1)|(df["ai_rating10_num"]>10)).sum()))
print("\nTop raw values of `ai_rating_10`:")
print(df[AI_R10].value_counts(dropna=False).head(15))

# ==========================================
# 3) Check ai_sentiment_5 <-> ai_rating_10
# ==========================================
df["ai_sent5_from_rating10"] = df["ai_rating10_num"].apply(rating10_to_sent5)
df["ai_sentiment_5_clean"] = df[AI5].map(norm5)

mask = df["ai_sent5_from_rating10"].notna() & df["ai_sentiment_5_clean"].notna()
match_rate = (df.loc[mask, "ai_sent5_from_rating10"] == df.loc[mask, "ai_sentiment_5_clean"]).mean()

print("\nAI hard (ai_sentiment_5) vs binned ai_rating_10 match rate:", round(match_rate, 6))
if match_rate < 1.0:
    bad = df.loc[mask].copy()
    bad = bad[bad["ai_sent5_from_rating10"] != bad["ai_sentiment_5_clean"]]
    print("Mismatches:", len(bad))
    display(bad[[AI_R10, "ai_sent5_from_rating10", AI5, "review"]].head(20))

# ==========================================
# 4) Confirm disagreement outcome definition
# ==========================================
df["human_sent5_clean"] = df[HUMAN5].map(norm5)
mask2 = df["human_sent5_clean"].notna() & df["ai_sentiment_5_clean"].notna()

df["disagree"] = (df["human_sent5_clean"] != df["ai_sentiment_5_clean"]).astype(int)

print("\nDisagreement rate (human 5 vs AI hard 5):",
      df.loc[mask2, "disagree"].mean().round(4),
      "| N =", int(mask2.sum()))

# quick crosstab sanity
print("\nCrosstab (human vs AI hard) top-left sanity:")
display(pd.crosstab(df.loc[mask2, "human_sent5_clean"], df.loc[mask2, "ai_sentiment_5_clean"]).reindex(
    index=["very_negative","negative","neutral","positive","very_positive"],
    columns=["very_negative","negative","neutral","positive","very_positive"]
))

Human rating_num summary:
count    28755.000000
mean         6.829108
std          3.331504
min          1.000000
25%          4.000000
50%          8.000000
75%         10.000000
max         10.000000
Name: rating_num, dtype: float64

Human rating_num invalid (NaN or outside 1..10): 0

Top raw values of `rating` (to confirm it's the original column):
rating
10    8863
9     4609
1     4103
8     3181
5     1609
7     1600
3     1332
2     1320
6     1138
4     1000
Name: count, dtype: int64

Date col dtype: object

AI ai_rating_10 summary:
count    28755.000000
mean         5.865902
std          2.887162
min          1.000000
25%          3.000000
50%          6.000000
75%          8.000000
max         10.000000
Name: ai_rating10_num, dtype: float64

AI ai_rating_10 invalid (NaN or outside 1..10): 0

Top raw values of `ai_rating_10`:
ai_rating_10
9.0     4561
8.0     4503
2.0     3226
7.0     3144
1.0     2510
6.0     2497
3.0     2187
10.0    2106
4.0     2077
5.0     1944
Name: coun

ai_sentiment_5_clean,very_negative,negative,neutral,positive,very_positive
human_sent5_clean,,,,,
very_negative,4218,1063,112,22,8
negative,859,1128,313,30,2
neutral,414,1173,927,215,18
positive,123,606,1692,1940,420
very_positive,122,294,1397,5440,6219


In [ ]:
ct = pd.crosstab(df.loc[mask2, "human_sent5_clean"], df.loc[mask2, "ai_sentiment_5_clean"])
ct = ct.reindex(index=["very_negative","negative","neutral","positive","very_positive"],
                columns=["very_negative","negative","neutral","positive","very_positive"],
                fill_value=0)
display(ct)

ai_sentiment_5_clean,very_negative,negative,neutral,positive,very_positive
human_sent5_clean,,,,,
very_negative,4218,1063,112,22,8
negative,859,1128,313,30,2
neutral,414,1173,927,215,18
positive,123,606,1692,1940,420
very_positive,122,294,1397,5440,6219


In [ ]:
PROB_COLS = ["ai_prob_very_negative","ai_prob_negative","ai_prob_neutral","ai_prob_positive","ai_prob_very_positive"]
P = df[PROB_COLS].astype(float)

# any negatives?
print("Any prob < 0:", (P < 0).any().any())

row_sum = P.sum(axis=1)
print("Row-sum summary:")
print(row_sum.describe())
print("Rows far from 1.0 (abs > 1e-3):", int((row_sum.sub(1).abs() > 1e-3).sum()))

Any prob < 0: False
Row-sum summary:
count    2.875500e+04
mean     1.000000e+00
std      1.436289e-16
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
dtype: float64
Rows far from 1.0 (abs > 1e-3): 0


In [ ]:
pmax = P.max(axis=1)
entropy = -(P.clip(1e-12,1).to_numpy() * np.log(P.clip(1e-12,1).to_numpy())).sum(axis=1) / np.log(5)

print("corr(entropy_norm, pmax) =", np.corrcoef(entropy, pmax)[0,1])

corr(entropy_norm, pmax) = -0.9282254160048259


In [ ]:
import numpy as np
import pandas as pd

HUMAN_COL   = "sentiment_5"
AI_HARD_COL = "ai_sentiment_5"

PROB_COLS = [
    "ai_prob_very_negative",
    "ai_prob_negative",
    "ai_prob_neutral",
    "ai_prob_positive",
    "ai_prob_very_positive",
]

def norm5(x):
    if pd.isna(x): return np.nan
    return (str(x).strip().lower().replace(" ", "_").replace("-", "_"))

def defensive_renorm(p, eps=1e-12):
    p = np.clip(p, eps, 1.0)
    return p / p.sum(axis=1, keepdims=True)

def entropy_norm(p):
    K = p.shape[1]
    H = -np.sum(p * np.log(p), axis=1)
    return H / np.log(K)

def make_df2(df):
    df2 = df.copy()

    # --- clean labels ---
    df2["human_sent5_clean"] = df2[HUMAN_COL].map(norm5)
    df2["ai_sentiment_5_clean"] = df2[AI_HARD_COL].map(norm5)

    # --- outcome: disagree (hard vs human) ---
    mask = df2["human_sent5_clean"].notna() & df2["ai_sentiment_5_clean"].notna()
    df2.loc[mask, "disagree"] = (df2.loc[mask, "human_sent5_clean"] != df2.loc[mask, "ai_sentiment_5_clean"]).astype(int)

    # --- predictors: review length ---
    if "review" in df2.columns:
        df2["review_len"] = df2["review"].astype(str).str.len()
    else:
        df2["review_len"] = np.nan

    # --- predictors: rating numeric (from human rating column "rating") ---
    # If you already have rating_num, this will just overwrite consistently.
    if "rating_num" in df2.columns:
        df2["rating_num"] = pd.to_numeric(df2["rating_num"], errors="coerce")
    elif "rating" in df2.columns:
        df2["rating_num"] = pd.to_numeric(df2["rating"], errors="coerce")
    else:
        df2["rating_num"] = np.nan

    # --- predictors: pmax + entropy_norm from probability vector ---
    missing = [c for c in PROB_COLS if c not in df2.columns]
    if len(missing) > 0:
        raise KeyError(f"Missing probability columns needed for entropy/pmax: {missing}")

    p_raw = df2[PROB_COLS].astype(float).to_numpy()
    p = defensive_renorm(p_raw)

    df2["pmax"] = p.max(axis=1)
    df2["entropy_norm"] = entropy_norm(p)

    return df2

df2 = make_df2(df)
print("N (with disagree defined):", int(df2["disagree"].notna().sum()))
print("Disagree rate:", float(df2.loc[df2["disagree"].notna(), "disagree"].mean()))

N (with disagree defined): 28755
Disagree rate: 0.49810467744740045


In [ ]:
# ============================================================
# FULL DROP-IN CELL: Rescaled Logit + robust SE via cov_type="HC3"
# - ORs are per 0.1 increase in entropy/pmax
# - ORs are per 100 characters increase in review length
# ============================================================

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

# ---------------------------
# CONFIG (edit if needed)
# ---------------------------
HUMAN_COL   = "sentiment_5"
AI_HARD_COL = "ai_sentiment_5"
REVIEW_COL  = "review"
RATING_COL  = "rating"

PROB_COLS = [
    "ai_prob_very_negative",
    "ai_prob_negative",
    "ai_prob_neutral",
    "ai_prob_positive",
    "ai_prob_very_positive",
]

# ---------------------------
# HELPERS
# ---------------------------
def norm5(x):
    if pd.isna(x):
        return np.nan
    return (str(x).strip()
            .lower()
            .replace(" ", "_")
            .replace("-", "_"))

def defensive_renorm(p, eps=1e-12):
    p = np.clip(p, eps, 1.0)
    s = p.sum(axis=1, keepdims=True)
    return p / s

def entropy_norm(p):
    K = p.shape[1]
    H = -np.sum(p * np.log(p), axis=1)
    return H / np.log(K)

def build_df2(df):
    df2 = df.copy()

    # clean labels
    df2["human_sent5_clean"] = df2[HUMAN_COL].map(norm5)
    df2["ai_sent5_clean"]    = df2[AI_HARD_COL].map(norm5)

    # outcome: disagree (based on HUMAN vs AI HARD labels)
    mask = df2["human_sent5_clean"].notna() & df2["ai_sent5_clean"].notna()
    df2.loc[mask, "disagree"] = (
        df2.loc[mask, "human_sent5_clean"] != df2.loc[mask, "ai_sent5_clean"]
    ).astype(int)

    # review length
    if REVIEW_COL in df2.columns:
        df2["review_len"] = df2[REVIEW_COL].astype(str).str.len()
    else:
        df2["review_len"] = np.nan

    # rating numeric
    if "rating_num" in df2.columns:
        df2["rating_num"] = pd.to_numeric(df2["rating_num"], errors="coerce")
    elif RATING_COL in df2.columns:
        df2["rating_num"] = pd.to_numeric(df2[RATING_COL], errors="coerce")
    else:
        df2["rating_num"] = np.nan

    # probs -> pmax + entropy_norm
    missing = [c for c in PROB_COLS if c not in df2.columns]
    if missing:
        raise KeyError(f"Missing probability columns: {missing}")

    p_raw = df2[PROB_COLS].astype(float).to_numpy()
    p = defensive_renorm(p_raw)

    df2["pmax"] = p.max(axis=1)
    df2["entropy_norm"] = entropy_norm(p)

    # ---------------------------
    # RESCALES for interpretability
    # ---------------------------
    # OR per 0.1 increase (instead of per 1.0)
    df2["entropy_per_0p1"] = df2["entropy_norm"] / 0.1
    df2["pmax_per_0p1"]    = df2["pmax"] / 0.1

    # OR per 100 characters (instead of per 1 char)
    df2["review_len_per_100"] = df2["review_len"] / 100.0

    return df2

def fit_disagree_logit(df2, predictor, cov_type="HC3"):
    """
    Fit: disagree ~ predictor + review_len_per_100 + rating_num
    """
    needed = ["disagree", predictor, "review_len_per_100", "rating_num"]
    d = df2[needed].dropna().copy()
    d["disagree"] = d["disagree"].astype(int)

    formula = f"disagree ~ {predictor} + review_len_per_100 + rating_num"
    res = smf.logit(formula, data=d).fit(disp=False, cov_type=cov_type)
    return res, d

def or_table_from_result(res, alpha=0.05):
    b = res.params
    se = res.bse
    p = res.pvalues
    ci = res.conf_int(alpha=alpha)
    ci.columns = ["coef_2.5%", "coef_97.5%"]

    out = pd.DataFrame({
        "coef": b,
        "se": se,
        "OR": np.exp(b),
        "OR_2.5%": np.exp(ci["coef_2.5%"]),
        "OR_97.5%": np.exp(ci["coef_97.5%"]),
        "p": p,
    })
    return out

# ============================================================
# RUN (ASSUMES df exists)
# ============================================================

df2 = build_df2(df)

print("N (with disagree defined):", int(df2["disagree"].notna().sum()))
print("Disagree rate:", float(df2.loc[df2["disagree"].notna(), "disagree"].mean()))

# ---- Model 1 (RESCALED): entropy_per_0p1 ----
res_ent, _ = fit_disagree_logit(df2, predictor="entropy_per_0p1", cov_type="HC3")
print("\nMODEL (entropy, rescaled): disagree ~ entropy_per_0p1 + review_len_per_100 + rating_num")
print(res_ent.summary())
display(or_table_from_result(res_ent))

# ---- Model 2 (RESCALED): pmax_per_0p1 ----
res_pmax, _ = fit_disagree_logit(df2, predictor="pmax_per_0p1", cov_type="HC3")
print("\nMODEL (pmax, rescaled): disagree ~ pmax_per_0p1 + review_len_per_100 + rating_num")
print(res_pmax.summary())
display(or_table_from_result(res_pmax))

N (with disagree defined): 28755
Disagree rate: 0.49810467744740045

MODEL (entropy, rescaled): disagree ~ entropy_per_0p1 + review_len_per_100 + rating_num
                           Logit Regression Results                           
Dep. Variable:               disagree   No. Observations:                28755
Model:                          Logit   Df Residuals:                    28751
Method:                           MLE   Df Model:                            3
Date:                Sat, 20 Dec 2025   Pseudo R-squ.:                  0.1972
Time:                        23:47:35   Log-Likelihood:                -16002.
converged:                       True   LL-Null:                       -19931.
Covariance Type:                  HC3   LLR p-value:                     0.000
                         coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept             -5.7285      0.

,coef,se,OR,OR_2.5%,OR_97.5%,p
Intercept,-5.728492,0.079768,0.003252,0.002781,0.003802,0.000000
entropy_per_0p1,0.669139,0.009795,1.952555,1.915428,1.990401,0.000000
review_len_per_100,-0.022027,0.005820,0.978214,0.967119,0.989437,0.000154
rating_num,0.185884,0.003989,1.204282,1.194904,1.213734,0.000000



MODEL (pmax, rescaled): disagree ~ pmax_per_0p1 + review_len_per_100 + rating_num
                           Logit Regression Results                           
Dep. Variable:               disagree   No. Observations:                28755
Model:                          Logit   Df Residuals:                    28751
Method:                           MLE   Df Model:                            3
Date:                Sat, 20 Dec 2025   Pseudo R-squ.:                  0.1868
Time:                        23:47:35   Log-Likelihood:                -16209.
converged:                       True   LL-Null:                       -19931.
Covariance Type:                  HC3   LLR p-value:                     0.000
                         coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept              3.2200      0.078     41.052      0.000       3.066       3.374
pmax_per_0p1          -0

,coef,se,OR,OR_2.5%,OR_97.5%,p
Intercept,3.219998,0.078437,25.028074,21.461561,29.187274,0.000000
pmax_per_0p1,-0.901906,0.014641,0.405795,0.394316,0.417609,0.000000
review_len_per_100,-0.006763,0.005748,0.993259,0.982132,1.004513,0.239363
rating_num,0.166311,0.003927,1.180940,1.171886,1.190064,0.000000


In [ ]:
import pandas as pd
import numpy as np
import re

def inspect_review_duplicates(df, review_col="review", top_k=20, show_examples=3, clean_ws=True):
    d = df.copy()

    # 1) Basic missingness
    na_n = int(d[review_col].isna().sum()) if review_col in d.columns else None
    print(f"Column: {review_col}")
    print(f"N rows: {len(d)}")
    print(f"Missing reviews: {na_n}")

    # 2) Choose raw vs cleaned key for grouping
    if clean_ws:
        key = (d[review_col]
               .fillna("")
               .astype(str)
               .str.replace(r"\s+", " ", regex=True)   # collapse whitespace
               .str.strip())
        key_name = "review_key_cleanws"
    else:
        key = d[review_col].fillna("").astype(str)
        key_name = "review_key_raw"

    d[key_name] = key

    # Watch out for literal "nan" strings (common artifact if someone did astype(str) earlier)
    nan_string_n = int((d[key_name].str.lower() == "nan").sum())
    empty_n = int((d[key_name] == "").sum())
    print(f'Empty string reviews after keying: {empty_n}')
    print(f'Literal "nan" string reviews after keying: {nan_string_n}')

    # 3) Duplicate counts
    vc = d[key_name].value_counts(dropna=False)
    unique_texts = int(vc.shape[0])
    dup_groups = int((vc >= 2).sum())
    dup_instances = int(vc[vc >= 2].sum())
    dup_pct = dup_instances / len(d)

    print("\n=== Duplicate summary (by exact match on key) ===")
    print(f"unique texts: {unique_texts}")
    print(f"duplicate groups (count>=2): {dup_groups}")
    print(f"duplicate instances (in those groups): {dup_instances}  ({dup_pct:.3%})")
    print(f"max duplicate group size: {int(vc.max())}")

    # 4) Show top duplicate groups + example texts
    top = vc[vc >= 2].head(top_k)
    if top.empty:
        print("\nNo duplicates found under this key definition.")
        return

    print(f"\n=== Top {min(top_k, len(top))} duplicate groups ===")
    display(top.to_frame("count"))

    print("\n=== Example duplicate texts ===")
    for i, (txt, cnt) in enumerate(top.items(), start=1):
        print(f"\n[{i}] count = {cnt}")
        # show a snippet so it doesn’t flood output
        snippet = txt[:500] + (" ..." if len(txt) > 500 else "")
        print(snippet)

        # show a few row indices where it occurs
        idxs = d.index[d[key_name] == txt].tolist()[:show_examples]
        print(f"example row indices: {idxs}")

def compare_raw_vs_cleanws(df, review_col="review"):
    print("---- RAW grouping ----")
    inspect_review_duplicates(df, review_col=review_col, clean_ws=False, top_k=10, show_examples=2)
    print("\n\n---- CLEANED whitespace grouping ----")
    inspect_review_duplicates(df, review_col=review_col, clean_ws=True, top_k=10, show_examples=2)

# ====== RUN ONE of these ======
# 1) single view (cleaned whitespace by default)
# inspect_review_duplicates(df, review_col="review", clean_ws=True)

# 2) compare raw vs cleaned grouping (recommended)
compare_raw_vs_cleanws(df, review_col="review")

---- RAW grouping ----
Column: review
N rows: 28755
Missing reviews: 0
Empty string reviews after keying: 0
Literal "nan" string reviews after keying: 0

=== Duplicate summary (by exact match on key) ===
unique texts: 25882
duplicate groups (count>=2): 2869
duplicate instances (in those groups): 5742  (19.969%)
max duplicate group size: 4

=== Top 10 duplicate groups ===


,count
review_key_raw,
"""Good""",4
"""Good medicine.""",3
"""I have posted early before about my 1st &amp; 2nd week using Retin A 0.025 cream. My 3rd week was much better than my second week, some of my acne stared to calm down and my face felt less bumpy. During this week I started to use the cream 4 times a week because my cheeks were hurting. My 4th week was the most improvement. I started to use the cream every night again since my face had stop hurting. During the 4th week most of my pimples disappeared and my skin felt smoother, I still had some pimples popping up but not as much. Now my cheeks are pretty smooth with a couple pimples but my cheeks are really red from the scars that were left behind from the pimples. I&#039;m hoping they will slowly fade later. Will keep updating!""",3
"""I am almost done w/ my 1st pack, which is a new formulation that came out late last year. I did take FE microgestin 1/20 a few years ago &amp; stopped because that made me lose my libido, gain weight, and just feel dead inside. I decided to give it a shot again &amp; was given this as a generic alternative I like this better than microgestin, but I have been feeling extremely tired and unmotivated, experienced minor spotting during the 3rd week of hormone pills, and my vision becomes blurry from time to time. Not sure if the blurry vision is normal, but I hope that these symptoms subside after my body gets more used to it. Will leave another review in a few months.""",2
"""I went off of depo 2 years and 7 months ago. I can not have a period on my own, so I no longer ovulated. We have been seeing a fertility doctor for 7 months. I can only have a period and ovulate with medication, but we have still had no luck. The only thing they can find wrong with me is the shot. I&#039;m so upset I may never be able to have another child. Do not take this if you ever want to have children. That&#039;s not including the side effects during: severe weight gain, mood swings""",2
"""I&#039;ve been on Alesse twice, this time for 2 months so far. I&#039;ve had no problems at all. I have polycystic ovarian syndrome and it&#039;s fixed my period so far. My complexion is getting better, I&#039;ve had no unusual depression or anxiety and I haven&#039;t gained additional weight. \r\n\r\n""",2
"""I had mine put in on Sept 25. I was just coming off the depo shot and starting to spot. I had the implant put in and afterwards the area was extremely painful for about a week. I am more fair skinned and so they told me I would bruise more. As soon as the lydocain was administered i had already developed a small bruise. By that evening my arm looked like it had been hit with a softball. Since then the needle insertion point has continued to be very sensitive. It seems like my body is trying to possibly reject the implant and is push one end of it out and the skin is raised there. Here it is 6 weeks later this Friday and i&#039;m still spotting and now slightly starting to cramp. Giving more time...""",2
"""I got a prescription for trinessa about 5 months ago, mostly just to help with my acne, and so far so good. My breakouts last a shorter amount of time and are smaller in size. My period has also gotten shorter and lighter and I hardly get cramps at all. The only drawback is that in the first 2-3 months or so I would get dizzy, slight headaches, and a little more fatigued than usual. Other than that this pill works great. My acne hasn&#039;t completely gone away, but it does help.""",2
"""I switched from Yaz to Ortho Tri Cyclen Lo and I&#039;ve been on it for almost three years. I liked it for the most part, my period was very predictable every month with minor pain-no mood swings or weight gain. I do get yeast infections more often than I ever have before though and I did get breast tenderness from time to time. I have now been off it for almost three weeks because my hands and feet have been swelling up. I&#039;ve been tested for everything and so far all my blood w


=== Example duplicate texts ===

[1] count = 4
"Good"
example row indices: [13251, 15041]

[2] count = 3
"Good medicine."
example row indices: [14443, 14648]

[3] count = 3
"I have posted early before about my 1st &amp; 2nd week using Retin A 0.025 cream. My 3rd week was much better than my second week, some of my acne stared to calm down and my face felt less bumpy. During this week I started to use the cream 4 times a week because my cheeks were hurting. My 4th week was the most improvement. I started to use the cream every night again since my face had stop hurting. During the 4th week most of my pimples disappeared and my skin felt smoother, I still had some pi ...
example row indices: [817, 913]

[4] count = 2
"I am almost done w/ my 1st pack, which is a new formulation that came out late last year. I did take FE microgestin 1/20 a few years ago &amp; stopped because that made me lose my libido, gain weight, and just feel dead inside. I decided to give it a shot again &amp; was g

,count
review_key_cleanws,
"""Good""",4
"""Good medicine.""",3
"""I have posted early before about my 1st &amp; 2nd week using Retin A 0.025 cream. My 3rd week was much better than my second week, some of my acne stared to calm down and my face felt less bumpy. During this week I started to use the cream 4 times a week because my cheeks were hurting. My 4th week was the most improvement. I started to use the cream every night again since my face had stop hurting. During the 4th week most of my pimples disappeared and my skin felt smoother, I still had some pimples popping up but not as much. Now my cheeks are pretty smooth with a couple pimples but my cheeks are really red from the scars that were left behind from the pimples. I&#039;m hoping they will slowly fade later. Will keep updating!""",3
"""I am almost done w/ my 1st pack, which is a new formulation that came out late last year. I did take FE microgestin 1/20 a few years ago &amp; stopped because that made me lose my libido, gain weight, and just feel dead inside. I decided to give it a shot again &amp; was given this as a generic alternative I like this better than microgestin, but I have been feeling extremely tired and unmotivated, experienced minor spotting during the 3rd week of hormone pills, and my vision becomes blurry from time to time. Not sure if the blurry vision is normal, but I hope that these symptoms subside after my body gets more used to it. Will leave another review in a few months.""",2
"""I went off of depo 2 years and 7 months ago. I can not have a period on my own, so I no longer ovulated. We have been seeing a fertility doctor for 7 months. I can only have a period and ovulate with medication, but we have still had no luck. The only thing they can find wrong with me is the shot. I&#039;m so upset I may never be able to have another child. Do not take this if you ever want to have children. That&#039;s not including the side effects during: severe weight gain, mood swings""",2
"""I&#039;ve been on Alesse twice, this time for 2 months so far. I&#039;ve had no problems at all. I have polycystic ovarian syndrome and it&#039;s fixed my period so far. My complexion is getting better, I&#039;ve had no unusual depression or anxiety and I haven&#039;t gained additional weight. """,2
"""I had mine put in on Sept 25. I was just coming off the depo shot and starting to spot. I had the implant put in and afterwards the area was extremely painful for about a week. I am more fair skinned and so they told me I would bruise more. As soon as the lydocain was administered i had already developed a small bruise. By that evening my arm looked like it had been hit with a softball. Since then the needle insertion point has continued to be very sensitive. It seems like my body is trying to possibly reject the implant and is push one end of it out and the skin is raised there. Here it is 6 weeks later this Friday and i&#039;m still spotting and now slightly starting to cramp. Giving more time...""",2
"""I got a prescription for trinessa about 5 months ago, mostly just to help with my acne, and so far so good. My breakouts last a shorter amount of time and are smaller in size. My period has also gotten shorter and lighter and I hardly get cramps at all. The only drawback is that in the first 2-3 months or so I would get dizzy, slight headaches, and a little more fatigued than usual. Other than that this pill works great. My acne hasn&#039;t completely gone away, but it does help.""",2
"""I switched from Yaz to Ortho Tri Cyclen Lo and I&#039;ve been on it for almost three years. I liked it for the most part, my period was very predictable every month with minor pain-no mood swings or weight gain. I do get yeast infections more often than I ever have before though and I did get breast tenderness from time to time. I have now been off it for almost three weeks because my hands and feet have been swelling up. I&#039;ve been tested for everything and so far all my blood work 


=== Example duplicate texts ===

[1] count = 4
"Good"
example row indices: [13251, 15041]

[2] count = 3
"Good medicine."
example row indices: [14443, 14648]

[3] count = 3
"I have posted early before about my 1st &amp; 2nd week using Retin A 0.025 cream. My 3rd week was much better than my second week, some of my acne stared to calm down and my face felt less bumpy. During this week I started to use the cream 4 times a week because my cheeks were hurting. My 4th week was the most improvement. I started to use the cream every night again since my face had stop hurting. During the 4th week most of my pimples disappeared and my skin felt smoother, I still had some pi ...
example row indices: [817, 913]

[4] count = 2
"I am almost done w/ my 1st pack, which is a new formulation that came out late last year. I did take FE microgestin 1/20 a few years ago &amp; stopped because that made me lose my libido, gain weight, and just feel dead inside. I decided to give it a shot again &amp; was g

In [2]:
import numpy as np
import pandas as pd

# ----------------------------
# Utilities
# ----------------------------
def make_review_key(df, review_col="review", clean_ws=True, key_col="review_key"):
    d = df.copy()
    if review_col not in d.columns:
        raise KeyError(f"review_col='{review_col}' not found. Available cols: {list(d.columns)[:30]} ...")
    s = d[review_col].fillna("").astype(str)
    if clean_ws:
        s = s.str.replace(r"\s+", " ", regex=True).str.strip()
    d[key_col] = s
    return d

def guess_label_cols(df):
    # tries to find plausible label columns automatically
    candidates = []
    for c in df.columns:
        cl = c.lower()
        if any(k in cl for k in ["label", "rating", "score", "sentiment", "y_", "target"]):
            candidates.append(c)
    # remove obviously-non-label columns
    bad = set(["split", "id", "review", "text", "text_orig"])
    candidates = [c for c in candidates if c.lower() not in bad]
    return candidates

def _maj_share(s):
    vc = s.value_counts(dropna=False)
    return float(vc.iloc[0] / vc.sum()) if len(vc) else np.nan

def _entropy(s):
    vc = s.value_counts(dropna=False, normalize=True)
    p = vc.values
    return float(-(p * np.log(p + 1e-12)).sum())

# ----------------------------
# 1) Duplicate label consistency
# ----------------------------
def dup_label_consistency(
    df,
    review_col="review",
    label_cols=None,
    clean_ws=True,
    key_col="review_key",
):
    d = make_review_key(df, review_col=review_col, clean_ws=clean_ws, key_col=key_col)

    if label_cols is None:
        label_cols = guess_label_cols(d)
        if not label_cols:
            raise ValueError(
                "Could not auto-detect label columns. Pass label_cols=[...] explicitly."
            )

    missing = [c for c in label_cols if c not in d.columns]
    if missing:
        raise KeyError(f"label_cols missing in df: {missing}. Available cols: {list(d.columns)[:30]} ...")

    g = d.groupby(key_col, dropna=False)
    out = pd.DataFrame({"n": g.size()})

    for col in label_cols:
        out[f"{col}_nunique"] = g[col].nunique(dropna=False)
        out[f"{col}_majshare"] = g[col].apply(_maj_share)
        out[f"{col}_entropy"]  = g[col].apply(_entropy)

    out = out.reset_index()
    dup = out[out["n"] >= 2].copy()
    return d, out, dup, label_cols

# ----------------------------
# 2) Text-only ceiling for each label source
# ----------------------------
def text_only_ceiling(df, key_col, label_col):
    g = df.groupby(key_col, dropna=False)[label_col]
    best_correct = g.apply(lambda s: s.value_counts(dropna=False).max()).sum()
    return float(best_correct / len(df))

# ----------------------------
# 3) Majority label agreement between two sources (per unique text)
# ----------------------------
def group_majority(df, key_col, label_col):
    return df.groupby(key_col, dropna=False)[label_col].apply(
        lambda s: s.value_counts(dropna=False).idxmax()
    )

def majority_agreement(df, key_col, label_a, label_b):
    a = group_majority(df, key_col, label_a).rename(label_a)
    b = group_majority(df, key_col, label_b).rename(label_b)
    maj = pd.concat([a, b], axis=1).dropna()
    return float((maj[label_a] == maj[label_b]).mean()), int(len(maj))

# ----------------------------
# 4) Metadata variation within duplicate texts (optional)
# ----------------------------
def dup_metadata_variation(df, key_col, drug_col="drug_id", cond_col="cond_id", useful_col="useful_z"):
    for c in [drug_col, cond_col, useful_col]:
        if c not in df.columns:
            raise KeyError(f"'{c}' not found in df. Available cols: {list(df.columns)[:30]} ...")

    g = df.groupby(key_col, dropna=False)
    out = g.agg(
        n=(key_col, "size"),
        drug_nunique=(drug_col, "nunique"),
        cond_nunique=(cond_col, "nunique"),
        useful_nunique=(useful_col, "nunique"),
    ).reset_index()

    out = out[out["n"] >= 2].copy()
    out["drug_or_cond_varies"] = (out["drug_nunique"] > 1) | (out["cond_nunique"] > 1)
    return out

# ==========================================================
# RUN THIS SECTION (edit only review_col / label_cols if needed)
# ==========================================================

# 1) Build keys + duplicate consistency table
d, all_groups, dup_groups, detected_labels = dup_label_consistency(
    df,
    review_col="review",       # <-- change if your column isn't named "review"
    label_cols=None,           # <-- OR set explicitly, e.g. ["human_label","gpt_text_label","gpt_meta_label"]
    clean_ws=True,
    key_col="review_key_cleanws",
)

print("Detected label columns:", detected_labels)
print("\nDuplicate rate (by cleaned key):",
      (d["review_key_cleanws"].duplicated().mean()))

print("\nSummary over duplicate groups (n>=2):")
display(dup_groups.describe(include="all"))

# 2) Text-only ceilings for each detected label column
print("\n=== Text-only ceiling (upper bound) by label source ===")
for col in detected_labels:
    try:
        ce = text_only_ceiling(d, "review_key_cleanws", col)
        print(f"{col}: {ce:.4f}")
    except Exception as e:
        print(f"{col}: [skip] {e}")

# 3) Pairwise majority agreement across label sources (per unique text)
print("\n=== Majority agreement per unique text (pairwise) ===")
for i in range(len(detected_labels)):
    for j in range(i+1, len(detected_labels)):
        a, b = detected_labels[i], detected_labels[j]
        agree, n_texts = majority_agreement(d, "review_key_cleanws", a, b)
        print(f"{a} vs {b}: agreement={agree:.4f} over {n_texts} unique texts")

# 4) OPTIONAL: metadata variation within duplicate text (only if these cols exist)
meta_cols = {"drug_id", "cond_id", "useful_z"}
if meta_cols.issubset(set(d.columns)):
    mv = dup_metadata_variation(d, "review_key_cleanws", "drug_id", "cond_id", "useful_z")
    print("\n=== Duplicate texts: how often metadata differs? ===")
    print("Share of duplicate groups where drug or condition varies:",
          float(mv["drug_or_cond_varies"].mean()))
else:
    print("\n[Info] Skipping metadata-variation check: need columns drug_id, cond_id, useful_z.")

Detected label columns: ['rating', 'sentiment_5', 'ai_rating_10', 'ai_sentiment_5', 'ai_prob_very_negative', 'ai_prob_very_positive', 'review_key_cleanws']

Duplicate rate (by cleaned key): 0.0999130585985046

Summary over duplicate groups (n>=2):


,review_key_cleanws,n,rating_nunique,rating_majshare,rating_entropy,sentiment_5_nunique,sentiment_5_majshare,sentiment_5_entropy,ai_rating_10_nunique,ai_rating_10_majshare,...,ai_sentiment_5_entropy,ai_prob_very_negative_nunique,ai_prob_very_negative_majshare,ai_prob_very_negative_entropy,ai_prob_very_positive_nunique,ai_prob_very_positive_majshare,ai_prob_very_positive_entropy,review_key_cleanws_nunique,review_key_cleanws_majshare,review_key_cleanws_entropy
count,2869,2869.000000,2869.000000,2869.000000,2.869000e+03,2869.000000,2869.000000,2.869000e+03,2869.000000,2869.000000,...,2.869000e+03,2869.000000,2869.000000,2.869000e+03,2869.000000,2869.000000,2.869000e+03,2869.0,2869.0,2.869000e+03
unique,2869,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,"""​Hello everyone, I&#039;m your typical 18 yea...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,NaN,2.001394,1.002788,0.998751,1.867456e-03,1.001743,0.999216,1.162399e-03,1.392820,0.803968,...,1.374500e-01,1.376438,0.811839,2.609070e-01,1.590450,0.705066,4.091082e-01,1.0,1.0,-1.000089e-12
std,NaN,0.045718,0.052741,0.024124,3.540010e-02,0.041717,0.019232,2.791047e-02,0.490599,0.244203,...,2.763958e-01,0.484576,0.242233,3.358590e-01,0.492545,0.245893,3.410919e-01,0.0,0.0,5.453557e-26
min,NaN,2.000000,1.000000,0.500000,-1.000089e-12,1.000000,0.500000,-1.000089e-12,1.000000,0.250000,...,-1.000089e-12,1.000000,0.500000,-1.000089e-12,1.000000,0.500000,-1.000089e-12,1.0,1.0,-1.000089e-12
25%,NaN,2.000000,1.000000,1.000000,-1.000089e-12,1.000000,1.000000,-1.000089e-12,1.000000,0.500000,...,-1.000089e-12,1.000000,0.500000,-1.000089e-12,1.000000,0.500000,-1.000089e-12,1.0,1.0,-1.000089e-12
50%,NaN,2.000000,1.000000,1.000000,-1.000089e-12,1.000000,1.000000,-1.000089e-12,1.000000,1.000000,...,-1.000089e-12,1.000000,1.000000,-1.000089e-12,2.000000,0.500000,6.931472e-01,1.0,1.0,-1.000089e-12
75%,NaN,2.000000,1.000000,1.000000,-1.000089e-12,1.000000,1.000000,-1.000089e-12,2.000000,1.000000,...,-1.000089e-12,2.000000,1.000000,6.931472e-01,2.000000,1.000000,6.931472e-01,1.0,1.0,-1.000089e-12



=== Text-only ceiling (upper bound) by label source ===
rating: 0.9997
sentiment_5: 0.9998
ai_rating_10: 0.9608
ai_sentiment_5: 0.9802
ai_prob_very_negative: 0.9624
ai_prob_very_positive: 0.9411
review_key_cleanws: 1.0000

=== Majority agreement per unique text (pairwise) ===
rating vs sentiment_5: agreement=0.0000 over 25882 unique texts
rating vs ai_rating_10: agreement=0.2616 over 25882 unique texts
rating vs ai_sentiment_5: agreement=0.0000 over 25882 unique texts
rating vs ai_prob_very_negative: agreement=0.0000 over 25882 unique texts
rating vs ai_prob_very_positive: agreement=0.0000 over 25882 unique texts
rating vs review_key_cleanws: agreement=0.0000 over 25882 unique texts
sentiment_5 vs ai_rating_10: agreement=0.0000 over 25882 unique texts
sentiment_5 vs ai_sentiment_5: agreement=0.0000 over 25882 unique texts
sentiment_5 vs ai_prob_very_negative: agreement=0.0000 over 25882 unique texts
sentiment_5 vs ai_prob_very_positive: agreement=0.0000 over 25882 unique texts
sentime